# Raster Validation

Validates raster built-up datasets (TEMPO, GHSL Built-S/V/H, Google OBT) against reference building footprints.

Pipeline per city:
1. Tile the AOI into 1 km × 1 km cells
2. For each enabled raster candidate, read the year-specific file (`{city_slug}_{name}_{year}.tif`)
3. Rasterize reference footprints onto the candidate grid (fractional coverage via oversampling)
4. Compute tile-level binary (TP/FP/FN/F1) and area-based metrics
5. Save tile metrics, city summary, and figures to `outputs/`

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir("/content/drive/MyDrive/Gates Foundation/Building Dataset Validation/")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/Gates Foundation/Building Dataset Validation")
CONFIG_PATH  = PROJECT_ROOT / "configs/validation_configs.yaml"

# Set overwrite=True to re-run cities whose raster outputs already exist.
# When False (default), cities with an existing sentinel parquet are skipped automatically.
OVERWRITE = True

In [3]:
import sys, os
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [4]:
import logging
import yaml
from src.validator import UrbanValidator

for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)

with open(CONFIG_PATH) as f:
    _cfg_preview = yaml.safe_load(f)

raster_datasets = _cfg_preview.get("raster", {}).get("datasets", [])
enabled = [
    f"{d['name']}" + (f"_{d['year']}" if d.get("year") is not None else "")
    for d in raster_datasets
    if d.get("enabled", True)
]
print(f"Config: {CONFIG_PATH}")
print(f"Enabled raster datasets: {enabled}")

Config: /content/drive/MyDrive/Gates Foundation/Building Dataset Validation/configs/validation_configs.yaml
Enabled raster datasets: ['obt_2023', 'tempo_2023q4', 'ghsl_built_s_2020']


In [5]:
# Patch overwrite flag and instantiate the Validator.
# This reads the config and AOI tracker, resolves file paths,
# and logs how many city datasets are queued.
import tempfile

with open(CONFIG_PATH) as f:
    _cfg_patched = yaml.safe_load(f)
_cfg_patched.setdefault("output", {})["overwrite"] = OVERWRITE

_tmp = tempfile.NamedTemporaryFile(
    mode="w", suffix=".yaml", delete=False, dir=PROJECT_ROOT / "configs"
)
yaml.dump(_cfg_patched, _tmp)
_tmp.close()
_PATCHED_CONFIG_PATH = _tmp.name

v = UrbanValidator(_PATCHED_CONFIG_PATH)
print(f"\nCities queued: {len(v.datasets)}")
for ds in v.datasets:
    print(f"  {ds['id']}")

01:33:42  INFO      Validation tracker: 70 -> 70 suitable rows.
01:33:52  INFO      Loaded 70 dataset(s) for validation.
01:33:52  INFO      Loaded 70 dataset(s) for validation.



Cities queued: 70
  ant-curacao
  bgd-rohingya
  blz-burrell-boom
  bra-nova-sussuarana
  col-san-antonio-de-prado
  cvg-saint-vincent-grenadines
  dom-dominica
  gha-accra
  gha-aiyim-sraha
  gha-dansoman
  gha-nawuni
  gha-sawla-tuna
  gha-wa
  grd-grenada
  jam-kingston
  jam-saint-catherine
  jpn-ashiya-hama
  jpn-hiroshima
  jpn-iwate
  jpn-izu-oshima
  jpn-kashima
  jpn-numakunai
  ken-kakuma
  ken-kakuma-kalobeyei
  ken-mukuru
  ken-nairobi
  lbr-monrovia
  lby-almarj
  lby-bayda
  lby-darnah
  lby-susah
  lca-saint-lucia
  maf-saint-martin
  mmr-patheingyi-mandalay
  moz-de-maio
  moz-djonasse
  mwi-lilongwe
  mwi-mlowe
  ner-niame
  nga-ibadan
  phl-bagamanoc
  phl-barangay
  phl-catanduanes
  phl-juraojurao-anini-y-antique
  phl-pasig
  phl-poblacion-sagua-anini-y-antique
  phl-viga
  phl-visayas
  sle-cockle-bay-1
  sle-cockle-bay-2
  sle-freetown
  sle-kolleh
  sle-kroo-bay-1
  ssd-juba
  swz-nhlangano
  sxm-sint-maarten
  tjk-artuch
  tjk-nomandiyon
  tjk-tavishi-bolo
  t

In [6]:
# Preview: show which raster files will be looked up for each city × dataset combination.
# Files that don't exist on disk are flagged so you can fix paths before running.
import pandas as pd
from pathlib import Path

data_dir = Path(_cfg_patched.get("root_dir", str(PROJECT_ROOT))) / _cfg_patched.get("data_dir", "data/01_raw")

preview_rows = []
for ds in v.datasets:
    city_slug    = ds["id"].lower()
    city_slug_us = city_slug.replace("-", "_")
    rast_dir     = data_dir / ds["id"] / "raster"

    for cand in _cfg_patched.get("raster", {}).get("datasets", []):
        if not cand.get("enabled", True):
            continue
        ds_name = cand["name"].replace("-", "_")
        year    = cand.get("year")
        if year is not None:
            fpath = rast_dir / f"{city_slug_us}_{ds_name}_{year}.tif"
        else:
            matches = sorted(rast_dir.glob(f"{city_slug_us}_{ds_name}*.tif"))
            fpath   = matches[0] if matches else rast_dir / f"{city_slug_us}_{ds_name}_?.tif"
        preview_rows.append({
            "city":    ds["id"],
            "dataset": f"{ds_name}_{year}" if year is not None else ds_name,
            "file":    fpath.name,
            "exists":  fpath.exists(),
        })

preview_df = pd.DataFrame(preview_rows)
missing = preview_df[~preview_df["exists"]]
print(f"Total city × dataset pairs: {len(preview_df)}")
print(f"Missing files:              {len(missing)}")
display(preview_df)

Total city × dataset pairs: 210
Missing files:              30


,city,dataset,file,exists
0,ant-curacao,obt_2023,ant_curacao_obt_2023.tif,True
1,ant-curacao,tempo_2023q4,ant_curacao_tempo_2023q4.tif,True
2,ant-curacao,ghsl_built_s_2020,ant_curacao_ghsl_built_s_2020.tif,True
3,bgd-rohingya,obt_2023,bgd_rohingya_obt_2023.tif,True
4,bgd-rohingya,tempo_2023q4,bgd_rohingya_tempo_2023q4.tif,True
...,...,...,...,...
205,uga-nakamiro,tempo_2023q4,uga_nakamiro_tempo_2023q4.tif,True
206,uga-nakamiro,ghsl_built_s_2020,uga_nakamiro_ghsl_built_s_2020.tif,True
207,ukr-pulyny,obt_2023,ukr_pulyny_obt_2023.tif,False
208,ukr-pulyny,tempo_2023q4,ukr_pulyny_tempo_2023q4.tif,True


In [7]:
results = v.validate_raster()

# Clean up temp config
try:
    os.unlink(_PATCHED_CONFIG_PATH)
except Exception:
    pass

# Summary
summary = pd.DataFrame(
    [{"city": k, "status": "ok" if ok else "failed"} for k, ok in results.items()]
)
print(f"\nDone — {len(summary)} cities processed.\n")
display(summary.groupby("status")["city"].count().rename("count").to_frame())
display(summary)

01:34:04  INFO      ━━━━  ant-curacao (raster)  ━━━━
01:34:04  INFO      MEM [ant-curacao raster start] RSS = 343 MB
01:34:04  INFO      [ant-curacao] Raster CRS: EPSG:32619 | 57 tiles.
01:34:05  INFO      [ant-curacao] Reference buildings: 15054
01:34:05  INFO      [ant-curacao / obt_2023] Raster candidate: ant_curacao_obt_2023.tif
01:34:08  INFO      [ant-curacao / obt_2023] Raster tile metrics saved → raster_metrics_tiles_obt_2023.parquet
01:34:08  INFO      MEM [ant-curacao/obt_2023 raster done] RSS = 486 MB
01:34:08  INFO      [ant-curacao / tempo_2023q4] Raster candidate: ant_curacao_tempo_2023q4.tif
01:34:10  INFO      [ant-curacao / tempo_2023q4] Raster tile metrics saved → raster_metrics_tiles_tempo_2023q4.parquet
01:34:10  INFO      MEM [ant-curacao/tempo_2023q4 raster done] RSS = 486 MB
01:34:10  INFO      [ant-curacao / ghsl_built_s_2020] Raster candidate: ant_curacao_ghsl_built_s_2020.tif
01:34:11  INFO      [ant-curacao / ghsl_built_s_2020] Raster tile metrics saved → ras

[grenada_ref.geojson] fixing 1 invalid geometries...


01:36:17  WARNING   [grd-grenada] No raster tile metrics produced — skipping output.
01:36:17  INFO      ━━━━  jam-kingston (raster)  ━━━━
01:36:17  INFO      MEM [jam-kingston raster start] RSS = 1320 MB
01:36:17  INFO      [jam-kingston] Raster CRS: EPSG:32618 | 345 tiles.
01:36:22  INFO      [jam-kingston] Reference buildings: 122846
01:36:22  INFO      [jam-kingston / obt_2023] Raster candidate: jam_kingston_obt_2023.tif
01:36:41  INFO      [jam-kingston / obt_2023] Raster tile metrics saved → raster_metrics_tiles_obt_2023.parquet
01:36:41  INFO      MEM [jam-kingston/obt_2023 raster done] RSS = 1334 MB
01:36:41  INFO      [jam-kingston / tempo_2023q4] Raster candidate: jam_kingston_tempo_2023q4.tif
01:36:51  INFO      [jam-kingston / tempo_2023q4] Raster tile metrics saved → raster_metrics_tiles_tempo_2023q4.parquet
01:36:51  INFO      MEM [jam-kingston/tempo_2023q4 raster done] RSS = 1334 MB
01:36:51  INFO      [jam-kingston / ghsl_built_s_2020] Raster candidate: jam_kingston_ghs

[saint_lucia_ref.geojson] fixing 2 invalid geometries...


01:43:52  INFO      [lca-saint-lucia] Reference buildings: 74526
01:43:52  WARNING   [lca-saint-lucia / obt] No raster file found (pattern: lca_saint_lucia_obt_2023.tif).
01:43:52  WARNING   [lca-saint-lucia / tempo] No raster file found (pattern: lca_saint_lucia_tempo_2023q4.tif).
01:43:52  WARNING   [lca-saint-lucia / ghsl_built_s] No raster file found (pattern: lca_saint_lucia_ghsl_built_s_2020.tif).
01:43:52  WARNING   [lca-saint-lucia] No raster tile metrics produced — skipping output.
01:43:52  INFO      ━━━━  maf-saint-martin (raster)  ━━━━
01:43:52  INFO      MEM [maf-saint-martin raster start] RSS = 3665 MB
01:43:52  INFO      [maf-saint-martin] Raster CRS: EPSG:32620 | 89 tiles.
01:43:53  INFO      [maf-saint-martin] Reference buildings: 13216
01:43:53  INFO      [maf-saint-martin / obt_2023] Raster candidate: maf_saint_martin_obt_2023.tif
01:43:55  ERROR     [maf-saint-martin] Unhandled error during raster validation.
Traceback (most recent call last):
  File "/content/drive

...

[kolleh_hotosm.geojson] Dropped 2 null/empty geometries (576 → 574)


01:47:30  INFO      [sle-kolleh / obt_2023] Raster tile metrics saved → raster_metrics_tiles_obt_2023.parquet
01:47:30  INFO      MEM [sle-kolleh/obt_2023 raster done] RSS = 1428 MB
01:47:30  INFO      [sle-kolleh / tempo_2023q4] Raster candidate: sle_kolleh_tempo_2023q4.tif
01:47:30  INFO      [sle-kolleh / tempo_2023q4] Raster tile metrics saved → raster_metrics_tiles_tempo_2023q4.parquet
01:47:30  INFO      MEM [sle-kolleh/tempo_2023q4 raster done] RSS = 1428 MB
01:47:30  INFO      [sle-kolleh / ghsl_built_s_2020] Raster candidate: sle_kolleh_ghsl_built_s_2020.tif
01:47:30  INFO      [sle-kolleh / ghsl_built_s_2020] Raster tile metrics saved → raster_metrics_tiles_ghsl_built_s_2020.parquet
01:47:30  INFO      MEM [sle-kolleh/ghsl_built_s_2020 raster done] RSS = 1428 MB
01:47:30  INFO      [sle-kolleh] Raster city summary saved.
/content/drive/MyDrive/Gates Foundation/Building Dataset Validation/src/output.py:367: FutureWarning: 

Passing `palette` without assigning `hue` is deprecat


Done — 70 cities processed.



,count
status,
failed,23
ok,47


,city,status
0,ant-curacao,ok
1,bgd-rohingya,ok
2,blz-burrell-boom,ok
3,bra-nova-sussuarana,failed
4,col-san-antonio-de-prado,ok
...,...,...
65,uga-bugoye,failed
66,uga-kampala,failed
67,uga-kanara,failed
68,uga-nakamiro,failed
